# Module 08 — Snapshotting and Restoring Data Streams

Data streams are append-only log/metrics constructs backed by a sequence of hidden
indices (`.ds-<name>-<timestamp>-NNNNNN`). They look simple from the outside but
have several snapshot/restore behaviours that catch people out.

This module walks through the full lifecycle and deliberately surfaces every
significant gotcha. All behaviour is verified against **Elasticsearch 9.x**.

## What you'll learn

| Section | Topic |
|---------|-------|
| 1 | Create a data stream with a composable template |
| 2 | What does a snapshot of a data stream actually contain? |
| 3 | Basic snapshot → delete → restore cycle |
| 4 | **Gotcha** — without the composable template, rollover breaks after restore |
| 5 | **Gotcha** — data stream must not exist on the restore target |
| 6 | **Gotcha** — `include_global_state` and why it matters |
| 7 | **Gotcha** — renaming a data stream on restore is tricky |
| 8 | **Gotcha** — restoring backing indices directly loses the data stream |
| 9 | **Gotcha** — the write index after restore |
| 10 | **Gotcha** — Fleet/Elastic Agent managed streams |
| 11 | Cleanup |


In [1]:
from helpers import *

es = get_client()
wait_for_green(es)
register_fs_repo(es)

✓ Cluster health: yellow

✓ Repository 'training-fs-repo' registered at /usr/share/elasticsearch/snapshots

---
## 1. Create a Data Stream

A data stream requires a **composable index template** whose `index_patterns` matches
the stream name and that contains a `data_stream: {}` block.

```
component template  ──►  composable index template  ──►  data stream
```

In [2]:
heading('1. Setup — component template + composable index template + data stream')

# --- cleanup from any previous run ---
for ds in ['training-logs-app', 'training-logs-renamed']:
    try:
        es.indices.delete_data_stream(name=ds)
        info(f'Deleted data stream: {ds}')
    except Exception:
        pass

for tpl in ['training-logs-template', 'training-logs-settings']:
    try:
        es.indices.delete_index_template(name=tpl)
    except Exception:
        pass
    try:
        es.cluster.delete_component_template(name=tpl)
    except Exception:
        pass

# --- component template with mappings and settings ---
es.cluster.put_component_template(
    name='training-logs-settings',
    body={
        'template': {
            'settings': {
                'number_of_shards': 1,
                'number_of_replicas': 0,
            },
            'mappings': {
                'properties': {
                    '@timestamp': {'type': 'date'},
                    'message':    {'type': 'text'},
                    'level':      {'type': 'keyword'},
                }
            },
        }
    },
)
success('Component template created: training-logs-settings')

# --- composable index template ---
es.indices.put_index_template(
    name='training-logs-template',
    body={
        'index_patterns': ['training-logs-*'],
        'data_stream': {},                          # ← marks this as a data-stream template
        'composed_of': ['training-logs-settings'],
        'priority': 200,
    },
)
success('Composable index template created: training-logs-template')

# --- create the data stream by indexing documents ---
for i in range(10):
    es.index(
        index='training-logs-app',
        document={'@timestamp': f'2025-01-01T{i:02d}:00:00Z', 'message': f'log line {i}', 'level': 'INFO'},
    )
es.indices.refresh(index='training-logs-app')
count = es.count(index='training-logs-app')['count']
success(f'Data stream training-logs-app created with {count} documents')

───────────────────── 1. Setup — component template + composable index template + data stream ─────────────────────

ℹ Deleted data stream: training-logs-app

✓ Component template created: training-logs-settings

✓ Composable index template created: training-logs-template

✓ Data stream training-logs-app created with 10 documents

---
## 2. What Is in a Data Stream Snapshot?

When you snapshot a data stream, Elasticsearch stores:

| Stored in snapshot | Requires |
|--------------------|----------|
| All **backing indices** (`.ds-*`) | default — always included when data stream is listed |
| **Data stream membership** (which stream each backing index belongs to) | ES 9.x: stored in backing index metadata — included by default |
| **Composable index template** | `include_global_state: true` |
| **Component templates** | `include_global_state: true` |

> **ES 9.x behaviour:** the data stream can be reconstructed on restore even without
> `include_global_state: true`, because backing index metadata records the stream name.
> However, without the composable template the stream is degraded — rollover fails.
> Always use `include_global_state: true` for a complete, self-contained snapshot.


In [3]:
heading('2. Inspect the backing indices of the data stream')

ds_info = es.indices.get_data_stream(name='training-logs-app')
stream = ds_info['data_streams'][0]

info(f"Data stream name    : {stream['name']}")
info(f"Generation          : {stream['generation']}")
info(f"Status              : {stream['status']}")
info(f"Template            : {stream['template']}")
info(f"Backing indices     : {[idx['index_name'] for idx in stream['indices']]}")
info(f"Write index         : {stream['indices'][-1]['index_name']}")

──────────────────────────────── 2. Inspect the backing indices of the data stream ────────────────────────────────

ℹ Data stream name    : training-logs-app

ℹ Generation          : 1

ℹ Status              : GREEN

ℹ Template            : training-logs-template

ℹ Backing indices     : ['.ds-training-logs-app-2026.04.11-000001']

ℹ Write index         : .ds-training-logs-app-2026.04.11-000001

In [4]:
heading('2b. Take a snapshot (include_global_state=True — the correct way)')

delete_snapshot_if_exists(es, FS_REPO_NAME, 'ds-full-snap')
result = es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='ds-full-snap',
    body={
        'indices': ['training-logs-app'],   # data stream name works here
        'include_global_state': True,       # captures templates and data stream metadata
    },
    wait_for_completion=True,
)
snap = result['snapshot']
success(f"Snapshot state  : {snap['state']}")
info(f"Backing indices : {snap['indices']}")
info(f"Data streams    : {snap.get('data_streams', [])}")
info(f"Feature states  : {[fs['feature_name'] for fs in snap.get('feature_states', [])]}")

──────────────────────── 2b. Take a snapshot (include_global_state=True — the correct way) ────────────────────────

ℹ Deleted existing snapshot 'ds-full-snap'

✓ Snapshot state  : SUCCESS

ℹ Backing indices : ['.integration_knowledge-7', '.kibana_9.3.0_001', '.ml-annotations-000001', 
'.ds-training-logs-app-2026.04.11-000001', '.kibana_task_manager_9.3.0_001', '.security-profile-8', 
'.async-search', '.inference', '.apm-agent-configuration', '.kibana_search_solution_9.3.0_001', 
'.ml-inference-000005', '.kibana_ingest_9.3.0_001', '.ml-notifications-000002', '.security-7', 
'.kibana_analytics_9.3.0_001', '.ml-inference-native-000002', '.kibana_security_session_1', '.kibana_locks-000001',
'.secrets-inference', '.kibana_usage_counters_9.3.0_001', '.apm-custom-link', 
'.kibana_security_solution_9.3.0_001', '.kibana_alerting_cases_9.3.0_001']

ℹ Data streams    : ['training-logs-app']

ℹ Feature states  : ['fleet', 'async_search', 'security', 'kibana', 'inference_plugin', 'machine_learning']

---
## 3. Basic Snapshot → Delete → Restore Cycle

In [5]:
heading('3. Delete the data stream and restore it')

# Guard: ensure ds-full-snap exists (re-create if section 2b was skipped or failed)
try:
    es.snapshot.get(repository=FS_REPO_NAME, snapshot='ds-full-snap')
    info('ds-full-snap found')
except Exception:
    warn('ds-full-snap missing — re-creating from current data stream state')
    # Also ensure the data stream exists first
    try:
        es.indices.get_data_stream(name='training-logs-app')
    except Exception:
        warn('Data stream also missing — re-run from section 1')
        raise
    delete_snapshot_if_exists(es, FS_REPO_NAME, 'ds-full-snap')
    es.snapshot.create(
        repository=FS_REPO_NAME,
        snapshot='ds-full-snap',
        body={
            'indices': ['training-logs-app'],
            'include_global_state': True,
        },
        wait_for_completion=True,
    )
    success('ds-full-snap re-created')

pre_count = es.count(index='training-logs-app')['count']
info(f'Documents before delete: {pre_count}')

# Must delete the data stream before restoring into the same name
es.indices.delete_data_stream(name='training-logs-app')
info('Data stream deleted')

es.snapshot.restore(
    repository=FS_REPO_NAME,
    snapshot='ds-full-snap',
    body={
        'indices': ['training-logs-app'],
        'include_global_state': True,
    },
    wait_for_completion=True,
)

es.indices.refresh(index='training-logs-app')
post_count = es.count(index='training-logs-app')['count']
success(f'Documents after restore: {post_count}')

# Confirm it is a data stream, not a plain index
ds_check = es.indices.get_data_stream(name='training-logs-app')
success(f'Restored as data stream: {ds_check["data_streams"][0]["name"]}')


──────────────────────────────────── 3. Delete the data stream and restore it ─────────────────────────────────────

ℹ ds-full-snap found

ℹ Documents before delete: 10

ℹ Data stream deleted

✓ Documents after restore: 10

✓ Restored as data stream: training-logs-app

---
## 4. Gotcha — Composable Index Template Must Exist Before Rollover

In ES 9.x, a data stream can be **restored** even without a matching composable
index template — the backing index metadata stored in the snapshot is enough to
reconstruct the stream.

However, without the template the data stream is **degraded**: any attempt to
**roll over** to a new backing index fails with
`no matching index template found for data stream`.  This means:

- Manual rollovers fail immediately.
- ILM-triggered rollovers fail, stalling the policy.
- The stream can only ever write to the single restored backing index.

> **Rule of thumb:** always snapshot data streams with `include_global_state: True`
> so the template travels with the snapshot, or pre-create the template on the
> target cluster before restoring.


In [ ]:
heading('4. Demonstrate the missing-template failure — rollover breaks')

import warnings
from elasticsearch import ElasticsearchWarning

# Take a snapshot WITHOUT global state
delete_snapshot_if_exists(es, FS_REPO_NAME, 'ds-no-global-snap')
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', ElasticsearchWarning)
    es.snapshot.create(
        repository=FS_REPO_NAME,
        snapshot='ds-no-global-snap',
        body={
            'indices': ['training-logs-app'],
            'include_global_state': False,     # ← deliberately omit global state
        },
        wait_for_completion=True,
    )
    for w in caught:
        warn(f'ES warning: {w.message}')
success('Snapshot ds-no-global-snap taken (no global state)')

# Delete the data stream AND the template
es.indices.delete_data_stream(name='training-logs-app')
es.indices.delete_index_template(name='training-logs-template')

# Verify template is actually gone before proceeding
try:
    es.indices.get_index_template(name='training-logs-template')
    raise RuntimeError('Template still exists after deletion — cannot demonstrate gotcha')
except RuntimeError:
    raise
except Exception:
    success('Confirmed: composable template deleted')

# In ES 9.x the restore itself succeeds — backing index metadata is enough
es.snapshot.restore(
    repository=FS_REPO_NAME,
    snapshot='ds-no-global-snap',
    body={
        'indices': ['training-logs-app'],
        'include_global_state': False,
    },
    wait_for_completion=True,
)
success('Restore succeeded — ES 9.x reconstructs the stream from backing index metadata')

# Verify template is still absent after restore
try:
    es.indices.get_index_template(name='training-logs-template')
    raise RuntimeError('Template was restored from snapshot — snapshot included global state')
except RuntimeError:
    raise
except Exception:
    success('Confirmed: template still absent after restore')

# The stream is degraded — rollover fails without a template.
# On ES 9.x this is expected behaviour; on earlier versions the restore itself would have failed.
es_version = tuple(int(x) for x in es.info()['version']['number'].split('.')[:2])
es_version_str = '.'.join(str(x) for x in es_version)
try:
    es.indices.rollover(alias='training-logs-app')
    if es_version >= (9, 0):
        warn(f'Rollover unexpectedly succeeded on ES {es_version_str} — re-run from section 1 to reset state')
    else:
        info(f'Rollover succeeded on ES {es_version_str} — template handling differs pre-9.x')
except Exception as e:
    success(f'Rollover failed as expected on ES {es_version_str} (stream has no template): {e.info["error"]["reason"]}')

# Fix: re-create the template
es.indices.put_index_template(
    name='training-logs-template',
    body={
        'index_patterns': ['training-logs-*'],
        'data_stream': {},
        'composed_of': ['training-logs-settings'],
        'priority': 200,
    },
)
success('Re-created composable template')

# Rollover now works
es.indices.rollover(alias='training-logs-app')
success('Rollover succeeded after template was re-created')
es.indices.refresh(index='training-logs-app')
info(f'Doc count after rollover: {es.count(index="training-logs-app")["count"]}')


---
## 5. Gotcha — Data Stream Must Not Exist on the Restore Target

Unlike plain indices (which can be **closed** before restore), a data stream must be
**deleted** before you can restore it under the same name.  Attempting to restore over
an open data stream raises `resource_already_exists_exception`.

Practical consequence during disaster recovery:

- You must make a deliberate decision to delete the live stream before the restore.
- If you need a side-by-side comparison, **rename on restore** (section 7) is the
  only option — you cannot have two data streams with the same name.

In [7]:
heading('5. Demonstrate restore-over-existing-stream failure')

# training-logs-app exists from section 4 — do NOT delete it
try:
    es.indices.get_data_stream(name='training-logs-app')
    info('Data stream training-logs-app exists: True')
except Exception:
    warn('Data stream training-logs-app does not exist — re-run from cell 1')

try:
    es.snapshot.restore(
        repository=FS_REPO_NAME,
        snapshot='ds-full-snap',
        body={
            'indices': ['training-logs-app'],
            'include_global_state': True,
        },
        wait_for_completion=True,
    )
    warn('Unexpectedly succeeded')
except Exception as e:
    warn(f'Expected failure: {type(e).__name__}')
    console.print(f'  [red]{str(e)[:300]}[/red]')

info('Fix: delete the data stream first, then restore')

─────────────────────────────── 5. Demonstrate restore-over-existing-stream failure ───────────────────────────────

ℹ Data stream training-logs-app exists: True

⚠ Expected failure: ApiError

ApiError(500, 'snapshot_restore_exception', ' cannot restore index [.ds-training-logs-app-2026.04.11-000001] 
because an open index with same name already exists in the cluster. Either close or delete the existing index or 
restore the inde

ℹ Fix: delete the data stream first, then restore

---
## 6. Gotcha — `include_global_state` and What You Lose Without It

When `include_global_state` is `False` (the default):

| What is **missing** from the snapshot | Consequence on restore (ES 9.x) |
|--------------------------------------|----------------------------------|
| Composable index template | Restore succeeds, but **rollover fails** — stream is write-locked to the single restored backing index (section 4) |
| Component templates | Mappings and settings from the component template are not applied to future backing indices |
| Cluster-level metadata (persistent settings, legacy templates) | Must exist independently on the target cluster |

The `data_streams` field in the snapshot response shows whether data stream metadata
was explicitly captured. In ES 9.x the stream can still be reassembled from backing
index metadata, but the composable template is not included without global state.


In [8]:
heading('6. Show what the snapshot contains with/without global state')

for snap_name in ['ds-full-snap', 'ds-no-global-snap']:
    meta = es.snapshot.get(repository=FS_REPO_NAME, snapshot=snap_name)
    s = meta['snapshots'][0]
    console.print(f'\n[bold]{snap_name}[/bold]')
    console.print(f'  indices              : {s["indices"]}')
    console.print(f'  data_streams         : {s.get("data_streams", [])}')
    console.print(f'  feature_states count : {len(s.get("feature_states", []))}')

────────────────────────── 6. Show what the snapshot contains with/without global state ───────────────────────────

ds-full-snap

indices              : ['.integration_knowledge-7', '.kibana_9.3.0_001', '.ml-annotations-000001', 
'.ds-training-logs-app-2026.04.11-000001', '.kibana_task_manager_9.3.0_001', '.security-profile-8', 
'.async-search', '.inference', '.apm-agent-configuration', '.kibana_search_solution_9.3.0_001', 
'.ml-inference-000005', '.kibana_ingest_9.3.0_001', '.ml-notifications-000002', '.security-7', 
'.kibana_analytics_9.3.0_001', '.ml-inference-native-000002', '.kibana_security_session_1', '.kibana_locks-000001',
'.secrets-inference', '.kibana_usage_counters_9.3.0_001', '.apm-custom-link', 
'.kibana_security_solution_9.3.0_001', '.kibana_alerting_cases_9.3.0_001']

data_streams         : ['training-logs-app']

feature_states count : 6

ds-no-global-snap

indices              : ['.ds-training-logs-app-2026.04.11-000001']

data_streams         : ['training-logs-app']

feature_states count : 0

---
## 7. Gotcha — Renaming a Data Stream on Restore Is Tricky

With regular indices, `rename_pattern` / `rename_replacement` is straightforward.
With data streams, the rename applies to the **data stream name** (and Elasticsearch
derives the backing index names from it automatically).

The critical constraint: **the renamed data stream name must match an index template
that has `data_stream: {}`**.  If `training-logs-renamed` does not match any template,
the restore fails.

Our template pattern `training-logs-*` covers the rename target, so this works.
If it did not, you would need to create an additional template for the new name.

> Renaming lets you keep the live stream running while you inspect a point-in-time
> copy alongside it — the only safe way to do a side-by-side comparison.

In [ ]:
heading('7. Rename data stream on restore')

import re

# Get the data stream names from the snapshot so we don't hard-code them
snap_info = es.snapshot.get(repository=FS_REPO_NAME, snapshot='ds-full-snap')
stream_names = snap_info['snapshots'][0].get('data_streams', [])
info(f'Streams in snapshot: {stream_names}')

# Restore the first stream with a "-renamed" suffix.
# rename_pattern uses a Java regex capture group so the stream name drives both fields.
stream_name        = stream_names[0]
rename_pattern     = f'({re.escape(stream_name)})'   # capture the exact name
rename_replacement = '$1-renamed'                     # append suffix via back-reference

info(f'rename_pattern:     {rename_pattern}')
info(f'rename_replacement: {rename_replacement}')
info(f'Result stream name: {stream_name}-renamed')

try:
    es.indices.delete_data_stream(name=f'{stream_name}-renamed')
except Exception:
    pass

es.snapshot.restore(
    repository=FS_REPO_NAME,
    snapshot='ds-full-snap',
    body={
        'indices': [stream_name],
        'rename_pattern':     rename_pattern,
        'rename_replacement': rename_replacement,
        'include_global_state': False,                 # template already exists
    },
    wait_for_completion=True,
)

es.indices.refresh(index=f'{stream_name}-renamed')
renamed_count  = es.count(index=f'{stream_name}-renamed')['count']
original_count = es.count(index=stream_name)['count']

success('Renamed restore succeeded')
info(f'  {stream_name}         : {original_count} docs (still running)')
info(f'  {stream_name}-renamed : {renamed_count} docs (point-in-time copy)')

# Confirm both are data streams (not plain indices)
both = es.indices.get_data_stream(name=f'{stream_name}*')
for s in both['data_streams']:
    info(f'  Data stream: {s["name"]}  backing indices: {[i["index_name"] for i in s["indices"]]}')

es.indices.delete_data_stream(name=f'{stream_name}-renamed')


---
## 8. Gotcha — Restoring Backing Indices Directly Loses the Data Stream

It is tempting to restore the `.ds-*` backing indices directly (they look like ordinary
indices in a snapshot's `indices` list).  **This works but produces plain indices, not a
data stream.**

Consequences:
- You cannot use the data stream write path (`op_type: create`) against them.
- The data stream name does not exist — queries against it return nothing.
- New documents cannot be routed to the correct backing index.

Always restore using the **data stream name**, not the `.ds-*` backing index pattern.

In [10]:
heading('8. Restore backing indices directly — observe the loss of data stream structure')

# Find the backing index name from the snapshot
snap_meta = es.snapshot.get(repository=FS_REPO_NAME, snapshot='ds-full-snap')
backing_indices = [i for i in snap_meta['snapshots'][0]['indices'] if i.startswith('.ds-training-logs-app')]
info(f'Backing indices in snapshot: {backing_indices}')

if not backing_indices:
    warn('No backing indices found — check snapshot content')
else:
    backing_index = backing_indices[0]

    # Delete the data stream so we can restore the backing index under its original name
    es.indices.delete_data_stream(name='training-logs-app')
    info('Deleted data stream training-logs-app')

    # Restore only the backing index — NOT the data stream name
    es.snapshot.restore(
        repository=FS_REPO_NAME,
        snapshot='ds-full-snap',
        body={
            'indices': [backing_index],
            'include_global_state': False,
        },
        wait_for_completion=True,
    )

    # The backing index is queryable as a plain index
    doc_count = es.count(index=backing_index)['count']
    success(f'Backing index queryable as plain index: {doc_count} docs')

    # But the data stream does NOT exist — no data stream was reassembled
    try:
        es.indices.get_data_stream(name='training-logs-app')
        warn('Data stream was reassembled (unexpected on this ES version)')
    except Exception:
        warn('Data stream training-logs-app does NOT exist')
        warn('The backing index is an orphaned plain index — cannot write to it via the data stream API')

    # Clean up and restore the data stream properly for subsequent sections
    es.indices.delete(index=backing_index)
    es.snapshot.restore(
        repository=FS_REPO_NAME,
        snapshot='ds-full-snap',
        body={
            'indices': ['training-logs-app'],
            'include_global_state': True,
        },
        wait_for_completion=True,
    )
    es.indices.refresh(index='training-logs-app')
    success(f'Data stream restored for subsequent sections: {es.count(index="training-logs-app")["count"]} docs')


───────────────── 8. Restore backing indices directly — observe the loss of data stream structure ─────────────────

ℹ Backing indices in snapshot: ['.ds-training-logs-app-2026.04.11-000001']

ℹ Deleted data stream training-logs-app

✓ Backing index queryable as plain index: 10 docs

⚠ Data stream training-logs-app does NOT exist

⚠ The backing index is an orphaned plain index — cannot write to it via the data stream API

✓ Data stream restored for subsequent sections: 10 docs

---
## 9. Gotcha — The Write Index After Restore

After restoring a data stream, the most-recent backing index becomes the write index
exactly as it was at snapshot time. Two practical concerns:

**1. Documenting the data gap**  
Any documents indexed between the snapshot and the restore are lost. There is a gap
in `@timestamp` that is invisible unless you check the document count.

**2. Manually triggering a rollover**  
After a restore you may want to force a clean rollover so new writes go to a fresh
backing index, clearly separating pre- and post-restore data.

In [11]:
heading('9. Inspect write index after restore and force a rollover')

ds_info = es.indices.get_data_stream(name='training-logs-app')
stream = ds_info['data_streams'][0]
write_index_before = stream['indices'][-1]['index_name']
generation_before = stream['generation']

info(f'Write index before rollover  : {write_index_before}')
info(f'Generation before rollover   : {generation_before}')

# Force rollover to mark the boundary between restored and new data
rollover_result = es.indices.rollover(alias='training-logs-app')
new_write_index = rollover_result.get('new_index', '(unknown)')
success(f'Rollover triggered. New write index: {new_write_index}')

# Confirm new backing index is in place
ds_info_after = es.indices.get_data_stream(name='training-logs-app')
stream_after = ds_info_after['data_streams'][0]
info(f'Write index after rollover   : {stream_after["indices"][-1]["index_name"]}')
info(f'Generation after rollover    : {stream_after["generation"]}')
info(f'All backing indices          : {[i["index_name"] for i in stream_after["indices"]]}')

warn('Pre-restore backing indices are now read-only — new writes land in the fresh backing index')

──────────────────────────── 9. Inspect write index after restore and force a rollover ────────────────────────────

ℹ Write index before rollover  : .ds-training-logs-app-2026.04.11-000001

ℹ Generation before rollover   : 1

✓ Rollover triggered. New write index: .ds-training-logs-app-2026.04.11-000002

ℹ Write index after rollover   : .ds-training-logs-app-2026.04.11-000002

ℹ Generation after rollover    : 2

ℹ All backing indices          : ['.ds-training-logs-app-2026.04.11-000001', 
'.ds-training-logs-app-2026.04.11-000002']

⚠ Pre-restore backing indices are now read-only — new writes land in the fresh backing index

---
## 10. Gotcha — Fleet / Elastic Agent Managed Streams

Streams prefixed `logs-*`, `metrics-*`, and `traces-*` are commonly owned by
**Elastic Agent / Fleet**.  These differ from custom data streams in important ways:

| Concern | Detail |
|---------|--------|
| Templates managed by Fleet | The composable template and component templates are installed by Fleet integration packages, not by you. Restoring with `include_global_state: True` will **overwrite** the current templates with the snapshotted versions — which may be older. |
| Ingest pipelines | Fleet installs named ingest pipelines. If those pipelines are not present on the target, indexing after restore will fail. |
| `.fleet-*` system indices | Fleet state (agent policies, enrollment tokens) lives in `.fleet-*` system indices, not inside data streams. Restoring data streams without restoring Fleet system indices leaves you with data but broken management. |
| Integration version drift | If the Fleet integration was upgraded between snapshot and restore, the template and pipeline versions will mismatch. |

**Recommended pattern for Fleet-managed streams:**

1. Re-install all Fleet integrations on the target cluster first (re-creates templates and pipelines).
2. Restore only `indices: ['logs-*', 'metrics-*']` with `include_global_state: False`.
3. Restore `.fleet-*` separately if you need to preserve agent enrollments.
4. Do **not** restore the composable templates with global state — let Fleet own them.

In [12]:
heading('10. Inspect Fleet-managed data streams (read-only observation)')

try:
    all_streams = es.indices.get_data_stream(name='*')
    fleet_streams = [
        s['name'] for s in all_streams.get('data_streams', [])
        if any(s['name'].startswith(p) for p in ('logs-', 'metrics-', 'traces-', '.fleet'))
    ]
    if fleet_streams:
        warn(f'Fleet-managed data streams detected ({len(fleet_streams)}):')
        for name in fleet_streams[:10]:
            info(f'  {name}')
        if len(fleet_streams) > 10:
            info(f'  ... and {len(fleet_streams) - 10} more')
        warn('Do NOT restore these with include_global_state=True — let Fleet own the templates')
    else:
        info('No Fleet-managed data streams found on this cluster — safe to ignore for this training environment')
except Exception as e:
    warn(f'Could not list data streams: {e}')

───────────────────────── 10. Inspect Fleet-managed data streams (read-only observation) ──────────────────────────

ℹ No Fleet-managed data streams found on this cluster — safe to ignore for this training environment

---
## 11. Cleanup

In [13]:
heading('11. Cleanup')

for ds in ['training-logs-app', 'training-logs-renamed']:
    try:
        es.indices.delete_data_stream(name=ds)
        success(f'Deleted data stream: {ds}')
    except Exception:
        pass

for tpl in ['training-logs-template', 'training-logs-settings']:
    try:
        es.indices.delete_index_template(name=tpl)
        success(f'Deleted index template: {tpl}')
    except Exception:
        pass
    try:
        es.cluster.delete_component_template(name=tpl)
        success(f'Deleted component template: {tpl}')
    except Exception:
        pass

for snap in ['ds-full-snap', 'ds-no-global-snap']:
    delete_snapshot_if_exists(es, FS_REPO_NAME, snap)

success('Cleanup complete')

─────────────────────────────────────────────────── 11. Cleanup ───────────────────────────────────────────────────

✓ Deleted data stream: training-logs-app

✓ Deleted index template: training-logs-template

✓ Deleted component template: training-logs-settings

ℹ Deleted existing snapshot 'ds-full-snap'

ℹ Deleted existing snapshot 'ds-no-global-snap'

✓ Cleanup complete

---
## Summary of Gotchas

| # | Gotcha | Fix |
|---|--------|-----|
| 4 | Without the composable template, rollover fails after restore (ES 9.x: restore itself succeeds) | Always use `include_global_state: True` **or** pre-create the template on the target |
| 5 | Data stream must not exist on restore target | `DELETE /<data-stream>` before restoring under the same name |
| 6 | `include_global_state: False` omits templates — rollover breaks on the restored stream | Use `include_global_state: True` for complete data stream snapshots |
| 7 | Renamed data stream must match an existing index template | Ensure the target name is covered by a `data_stream: {}` template before renaming |
| 8 | Restoring `.ds-*` backing indices directly produces plain indices | Always use the data stream name in the `indices` list, not `.ds-*` patterns |
| 9 | Write index after restore reflects snapshot time; a data gap exists | Force a rollover after restore to establish a clean write boundary |
| 10 | Fleet-managed streams have template/pipeline ownership conflicts | Re-install integrations on the target first; restore data only with `include_global_state: False` |

**Next:** back to [07_advanced_topics.ipynb](07_advanced_topics.ipynb)
